In [12]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from evaluate import evaluate_bilstm_crf
from models.bilstm_crf import BiLSTM_CRF
from utils.bilstm_crf.data_utils import (
    read_conll_2,
    build_vocab,
    build_tag2idx,
    NERDataset,
    collate_fn
)

In [13]:
TRAIN_PATH = "../data/matscholar/train.txt"
VALID_PATH = "../data/matscholar/valid.txt"

EMBEDDING_DIM = 300
HIDDEN_DIM = 384
BATCH_SIZE = 15
EPOCHS = 50
LEARNING_RATE = 5e-4

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [14]:
train_sentence, train_tags = read_conll_2(TRAIN_PATH)
valid_sentence, valid_tags = read_conll_2(VALID_PATH)

word2idx = build_vocab(train_sentence)
tag2idx, idx2tag = build_tag2idx(train_tags)

In [15]:
train_data = NERDataset(train_sentence, train_tags, word2idx, tag2idx)
valid_data = NERDataset(valid_sentence, valid_tags, word2idx, tag2idx)

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

model = BiLSTM_CRF(
    vocab_size=len(word2idx),
    tag_to_ix=tag2idx,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

In [16]:
# =========================
# 5. 训练
# =========================
best_valid_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0.0
    total_train_sentences = 0

    # 训练阶段也改成整个 batch 直接计算 loss

    for sentences, tags, lengths in train_loader:
        sentences = sentences.to(DEVICE)
        tags = tags.to(DEVICE)
        lengths = lengths.to(DEVICE)

        optimizer.zero_grad()

        # 直接以整个 batch 为单位计算平均 loss
        # 内部会自动：
        # 1. 计算整个 batch 的发射分数
        # 2. 根据 lengths 构造 mask
        # 3. 调用 torchcrf 计算 batch 的负对数似然
        loss = model.neg_log_likelihood(sentences, tags, lengths)

        batch_size = sentences.size(0)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item() * batch_size
        total_train_sentences += batch_size

    avg_train_loss = total_train_loss / total_train_sentences

    # 每轮结束后在验证集评估
    valid_loss, valid_p, valid_r, valid_f1, _, _ = evaluate_bilstm_crf(
        model, valid_loader, idx2tag, DEVICE
    )

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Precision: {valid_p:.4f}")
    print(f"Valid Recall: {valid_r:.4f}")
    print(f"Valid F1: {valid_f1:.4f}")

    # 保存验证集上最好的模型
    if valid_f1 > best_valid_f1:
        best_valid_f1 = valid_f1

        torch.save({
            "model_state_dict": model.state_dict(),
            "word2idx": word2idx,
            "tag2idx": tag2idx
        }, "../models/bilstm_crf.pt")

        print("保存当前最佳模型")
    scheduler.step(valid_f1)


Epoch 1/50
Train Loss: 26.8914
Valid Loss: 17.8768
Valid Precision: 0.4691
Valid Recall: 0.2918
Valid F1: 0.3598
保存当前最佳模型

Epoch 2/50
Train Loss: 14.7811
Valid Loss: 12.4377
Valid Precision: 0.5734
Valid Recall: 0.5690
Valid F1: 0.5712
保存当前最佳模型

Epoch 3/50
Train Loss: 10.5983
Valid Loss: 10.5205
Valid Precision: 0.6637
Valid Recall: 0.6126
Valid F1: 0.6371
保存当前最佳模型

Epoch 4/50
Train Loss: 8.2036
Valid Loss: 9.0971
Valid Precision: 0.6760
Valid Recall: 0.6559
Valid F1: 0.6658
保存当前最佳模型

Epoch 5/50
Train Loss: 6.5654
Valid Loss: 8.5267
Valid Precision: 0.6810
Valid Recall: 0.6961
Valid F1: 0.6885
保存当前最佳模型

Epoch 6/50
Train Loss: 5.3272
Valid Loss: 8.3169
Valid Precision: 0.6987
Valid Recall: 0.7000
Valid F1: 0.6994
保存当前最佳模型

Epoch 7/50
Train Loss: 4.3728
Valid Loss: 8.2262
Valid Precision: 0.7048
Valid Recall: 0.7194
Valid F1: 0.7120
保存当前最佳模型

Epoch 8/50
Train Loss: 3.5925
Valid Loss: 7.9090
Valid Precision: 0.7162
Valid Recall: 0.7276
Valid F1: 0.7219
保存当前最佳模型

Epoch 9/50
Train Loss: 2.